# Results for model: mistralai/mistral-medium-3-instruct

In [1]:
import polars as pl
import xgboost as xgb
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# Load data
df = pl.read_parquet('./jane_street_data/train.parquet')

# Constants
TOP_QUANTILE = 0.85
rolling_batch_size = 10

# Group by date_id: changepoint detection
def _groupby_date_id(df: pl.DataFrame) -> pl.DataFrame:
    return (df
             .group_by('date_id', maintain_order=True)
             .agg(
                pl.col('feature_00').alias('tmp_feature'),
                pl.col('responder_6').alias('tmp_responder'))
            .with_columns([
                (pl.col('tmp_responder') - pl.col('tmp_feature')).alias('diff'),
                (pl.col('tmp_responder') > pl.col('tmp_feature')).alias('greater')])
            .with_columns([
                pl.col(['date_id', 'tmp_feature', 'tmp_responder', 'diff', 'greater']).shift(-1).suffix("_shifted"),
                pl.col(['date_id', 'tmp_feature', 'tmp_responder', 'diff', 'greater']).shift(1, fill_null=pl.lit(None)).suffix("_lagged")])
            )

# Feature engineering
def create_features(df: pl.DataFrame) -> pl.DataFrame:
    df = _groupby_date_id(df)

    # Create quantile of feature_00 relative to responder_6 across rolling batches
    df = df.sort(['date_id', 'tmp_feature'], descending=[False, True])

    df = (df
          .with_columns([
            pl.col('date_id').cast(pl.Int32).alias("date_id_int")
        ])
        .with_columns([
            pl.col('tmp_feature').rolling.apply(lambda x: x.quantile(TOP_QUANTILE), window_size=rolling_batch_size).over("date_id_int").fill_null(0).cast(pl.Float32).alias('feature_quantile')
        ])
        .with_columns([
            pl.when(pl.col("tmp_feature") >= pl.col("feature_quantile"))
            .then(pl.lit(True))
            .otherwise(pl.lit(False))
            .alias("is_in_top_quantile")
        ])
    )

    # Create features that might represent dynamic characteristics of pairs procuded within a window-size
    df = (df
          .with_columns([
              pl.col("feature_quantile").cast(pl.Float32).alias("quantile").rolling_mean(window_size=rolling_batch_size).over("date_id_int").fill_null(0).alias("quantile_mean"),
              pl.col("feature_quantile").cast(pl.Float32).alias("quantile").rolling_std(window_size=rolling_batch_size).over("date_id_int").fill_null(0).alias("quantile_std"),
              pl.col("feature_quantile").cast(pl.Float32).alias("quantile").rolling_max(window_size=rolling_batch_size).over("date_id_int").fill_null(0).alias("quantile_max"),
              pl.col("feature_quantile").cast(pl.Float32).alias("quantile").rolling_min(window_size=rolling_batch_size).over("date_id_int").fill_null(0).alias("quantile_min"),
          ])
    )

    # Join
    joined_df = (df
                 .join(df.select(pl.all().suffix('_shifted')), on="date_id", suffix='shifted', how="left")
                 .join(df.select(pl.all().suffix('_lagged')), on="date_id", suffix='lagged', how="left")
                )

    return joined_df

# Prepare data for training
def prepare_data(df: pl.DataFrame) -> tuple:
    df = create_features(df)
    df = df.fill_null(0)

    # Select top quantile data
    df = df.filter(pl.col("is_in_top_quantile") == True)

    X_ = (df
          .drop(['tmp_feature', 'tmp_responder',
                 'tmp_feature_shifted', 'tmp_responder_shifted',
                 'tmp_feature_lagged', 'tmp_responder_lagged',
                 'diff', 'diff_shifted', 'diff_lagged',
                 'greater', 'greater_shifted', 'greater_lagged',
                 'is_in_top_quantile',
                 'date_id', 'feature_quantile'
                 ])
          )

    y_ = (df['responder_6']).drop_nulls()

    X, y = X_.to_pandas(), y_.to_pandas()

    return X, y

X, y = prepare_data(df)

# Split data into train and test sets
X_train, X

FileNotFoundError: The system cannot find the path specified. (os error 3): ./jane_street_data/train.parquet